<a href="https://colab.research.google.com/github/paskasiregar/adaptive-xai-credit/blob/main/notebooks/home-credit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1: Get data, auth, lib

In [ ]:
!pip install kagglehub -q

In [ ]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
path = kagglehub.competition_download('home-credit-default-risk')
print("Files downloaded to:", path)

100%|██████████| 688M/688M [00:08<00:00, 87.6MB/s]

Extracting files...


Files downloaded to: /root/.cache/kagglehub/competitions/home-credit-default-risk


In [ ]:
import os
os.listdir(path)

['application_test.csv',
 'POS_CASH_balance.csv',
 'previous_application.csv',
 'bureau.csv',
 'bureau_balance.csv',
 'installments_payments.csv',
 'sample_submission.csv',
 'application_train.csv',
 'credit_card_balance.csv',
 'HomeCredit_columns_description.csv']

In [ ]:
import pandas as pd

df = pd.read_csv(f"{path}/application_train.csv")
print(df.shape)
df.head()

(307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


# 2: Inspection + Cleaning

## 2a: Inspection

In [ ]:
# Check class imbalance
print("Target distribution:")
print(df['TARGET'].value_counts())
print()

# Check missing values - top 20 worst columns
print("Top 20 columns by missing value %:")
print((df.isnull().mean() * 100).sort_values(ascending=False).head(20).round(2))

Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Top 20 columns by missing value %:
COMMONAREA_AVG              69.87
COMMONAREA_MODE             69.87
COMMONAREA_MEDI             69.87
NONLIVINGAPARTMENTS_MEDI    69.43
NONLIVINGAPARTMENTS_MODE    69.43
NONLIVINGAPARTMENTS_AVG     69.43
FONDKAPREMONT_MODE          68.39
LIVINGAPARTMENTS_AVG        68.35
LIVINGAPARTMENTS_MEDI       68.35
LIVINGAPARTMENTS_MODE       68.35
FLOORSMIN_MODE              67.85
FLOORSMIN_AVG               67.85
FLOORSMIN_MEDI              67.85
YEARS_BUILD_AVG             66.50
YEARS_BUILD_MODE            66.50
YEARS_BUILD_MEDI            66.50
OWN_CAR_AGE                 65.99
LANDAREA_MEDI               59.38
LANDAREA_AVG                59.38
LANDAREA_MODE               59.38
dtype: float64


"Check missing values - top 20 worst columns" -> this is to see which columns have the most missing data, ranked worst to best.

For example, `COMMONAREA_AVG = 69.87` means 69.87% of applicants have no value in that column. Nearly 7 out of 10 rows are blank for that feature.

### 2a.1: Verifying the `COMMONAREA_AVG`

In [ ]:
desc = pd.read_csv(f"{path}/HomeCredit_columns_description.csv", encoding='latin-1')
desc

,Unnamed: 0,Table,Row,Description,Special
0,1,application_{train|test}.csv,SK_ID_CURR,ID of loan in our sample,NaN
1,2,application_{train|test}.csv,TARGET,Target variable (1 - client with payment diffi...,NaN
2,5,application_{train|test}.csv,NAME_CONTRACT_TYPE,Identification if loan is cash or revolving,NaN
3,6,application_{train|test}.csv,CODE_GENDER,Gender of the client,NaN
4,7,application_{train|test}.csv,FLAG_OWN_CAR,Flag if the client owns a car,NaN
...,...,...,...,...,...
214,217,installments_payments.csv,NUM_INSTALMENT_NUMBER,On which installment we observe payment,NaN
215,218,installments_payments.csv,DAYS_INSTALMENT,When the installment of previous credit was su...,time only relative to the application
216,219,installments_payments.csv,DAYS_ENTRY_PAYMENT,When was the installments of previous credit p...,time only relative to the application
217,220,installments_payments.csv,AMT_INSTALMENT,What was the prescribed installment amount of ...,NaN


In [ ]:
desc[desc['Row'].isin(['COMMONAREA_AVG'])]

,Unnamed: 0,Table,Row,Description,Special
48,51,application_{train|test}.csv,COMMONAREA_AVG,Normalized information about building where th...,normalized


In [ ]:
df[['SK_ID_CURR', 'COMMONAREA_AVG']]

,SK_ID_CURR,COMMONAREA_AVG
0,100002,0.0143
1,100003,0.0605
2,100004,NaN
3,100006,NaN
4,100007,NaN
...,...,...
307506,456251,0.0202
307507,456252,0.0022
307508,456253,0.0123
307509,456254,NaN


This is to see how many `COMMONAREA_AVG` values are NaN (missing).

## 2b: Cleaning

In [ ]:
threshold = 0.4
df_clean = df[df.columns[df.isnull().mean() < threshold]] #df_clean is one big table: 307,511 rows, 73 columns. It contains everything: the inputs (income, loan amount, employment length, etc) AND the answer (did this person default or not).

# Need to split them apart because the model's job is to learn: "given the inputs, predict the answer." If the answer is sitting inside the inputs, the model just reads it directly and learns nothing.

print("Columns before:", df.shape[1])
print("Columns after:", df_clean.shape[1])
print("Columns dropped:", df.shape[1] - df_clean.shape[1])

Columns before: 122
Columns after: 73
Columns dropped: 49


Drop columns missing more than 40% of values.

**What this step does:** Identifies and removes columns with too much missing data, then prepares the dataframe for model training.

**Note on property columns:** The columns being dropped are granular building measurements (common area size, floor counts, years built) -- not basic ownership flags. `FLAG_OWN_REALTY` and `FLAG_OWN_CAR` are separate, complete columns that stay in the data.

## 2c: Separate target, encode, fill gaps

In [ ]:
# Separate target from features
X = df_clean.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df_clean['TARGET']

# Encode categorical columns (convert text categories to numbers)
X = pd.get_dummies(X)

# Fill remaining missing values with column median
X = X.fillna(X.median())

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing values remaining:", X.isnull().sum().sum())

X shape: (307511, 183)
y shape: (307511,)
Missing values remaining: 0


### Line-by-line exp

```
X = df_clean.drop(columns=['TARGET', 'SK_ID_CURR'])
```
> `X` is the **inputs** -> everything the model is allowed to look at when making a prediction.
> `drop(columns=...)` -> removes columns from the table

So it's removing 2 things:
*   TARGET -- this is the answer (defaulted or not). The model cannot see this during training.
*   SK_ID_CURR -- this is just a loan ID number, like a row number. It has no predictive meaning. Keeping it would confuse the model.

**Result**: `X` is the full table minus those two columns. 71 columns remaining.


<center> --- </center>


```
y = df_clean['TARGET']
```
> `y` is the **answer** -> a single column of 307,511 values, each either 0 or 1.

**This is what the model is trying to predict**. During training it looks at a row in X, makes a guess, then checks against y to see if it was right. It does this hundreds of thousands of times and eventually gets better.

<center> --- </center>


**What `pd.get_dummies()` does:** Converts text columns (like `NAME_CONTRACT_TYPE`: "Cash loans" / "Revolving loans") into numeric columns the model can read. Each category becomes its own 0/1 column.

<center> --- </center>

```
X = X.fillna(X.median())
```
After dropping the high-missing columns, there are still some columns with a small number of missing values. Not enough to drop the whole column, but still blanks the model cannot handle.
> `fillna()` fills those blanks.

> `X.median()`: fill each blank with the **middle** value of that column.

For example, if `AMT_INCOME_TOTAL` has a few missing values and the median income across all applicants is 147,000 -> those blanks become 147,000.
It is not perfectly accurate (this is a guess), but it is a reasonable guess and it removes all blanks. The last line confirms this worked: Missing values remaining: 0.

**What `.fillna(X.median())` does:** For any remaining missing values, fills them with the middle value of that column. This is a pragmatic choice -- good enough for a proxy model used to generate SHAP values.

# 3: Split data

There are 307,511 applicants in X and y. We cannot use all of them to train the model, because then we would have nothing left to test it on. If we test on the same data we trained it on, the model looks artificially good. It has already seen those rows, so it is not a fair test.

> **The solution**: split the data into two separate groups before training. The model only ever sees the training set. The test set is locked away until evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (246008, 183)
Test set: (61503, 183)


### Line-by-line exp
> `test_size=0.2` -> 20% goes to testing, 80% goes to training.

> `random_state=42` -> The split is random.

Pandas shuffles the rows before dividing them. `random_state=42` fixes the shuffle so we get the same split every time we run the cell. Without it, every run would produce a slightly different split and the results would not be reproducible. The number 42 is arbitrary, any fixed number works.

> `stratify=y` -> The class imbalance: 282,686 non-defaulters vs 24,825 defaulters | roughly 11:1.

  Without stratify, the random shuffle might accidentally put most defaulters in the training set and very few in the test set -- or vice versa. That would make evaluation misleading.
  `stratify=y` tells pandas: whatever the ratio is in the full dataset, preserve that same ratio in both the training and test sets. So both sets end up with approximately 11:1 non-defaulters to defaulters.

# 4: Model config

XGBoost builds **decision trees** one at a time, sequentially. Each new tree looks at where the previous tree made mistakes and tries to correct them. After 100 trees, the final prediction is the combined result of all of them. This is called **gradient boosting**.

In [ ]:
import xgboost as xgb

# Calculate scale_pos_weight from your actual data
scale = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
print("scale_pos_weight:", round(scale, 2))

model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=scale,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

scale_pos_weight: 11.39


### Line-by-line exp
```
`n_estimators=100`-> How many trees to build.
```
100 trees is a reasonable starting point. Enough to learn meaningful patterns without taking too long to train. More trees generally means better accuracy up to a point, then it plateaus (?).

<center> --- </center>

```
max_depth=4 -> How deep each individual tree can grow.
```
Depth means how many yes/no questions it can ask before making a decision. A tree with depth 4 can ask 4 questions:
  ``` Is debt-to-income > 0.5?
    → Is employment length < 1 year?
      → Is income < 50,000?
        → Is loan amount > 200,000? → PREDICT DEFAULT
  ```
Deeper trees learn more complex patterns but also risk **memorising** the training data rather than learning general rules. Depth 4 is a conservative, stable choice.

<center> --- </center>


```
`learning_rate=0.1` -> How much each new tree is allowed to correct the previous one.
```
A lower value means each tree makes smaller corrections -- the model learns more slowly but more carefully. **0.1 is the standard starting point.**
If you imagine the model walking toward the correct answer, learning rate controls the size of each step.

<center> --- </center>

```
scale_pos_weight -> Corrects the 11:1 class imbalance.
```
Without it, the model would learn that predicting "repaid" for everyone gives it 91% accuracy -- technically high, but completely useless for identifying actual defaulters.
This var tells the model: treat each defaulter as if they are worth 11 non-defaulters. This forces it to pay attention to the minority class.
We calculate it directly from our data rather than hardcoding 11, because the exact ratio from the actual split is slightly different from the raw dataset ratio.

<center> --- </center>

```
random_state=42
```
Same reason as Step 3 -- fixes randomness so results are reproducible.

---

### What this step produces

A configured but untrained model object called model. Nothing has been learned yet. Step 5 is when the actual learning happens.

## What `scale_pos_weight` does

It tells the model: every time you encounter a defaulter during training, count it as 11.39 people instead of 1.

So the 24,825 defaulters now feel like 24,825 * 11.39 = ~282,000 people to the model. Roughly equal weight to the non-defaulters.

Now the model cannot ignore them. Missing a defaulter costs it just as much as missing a non-defaulter. So it is forced to actually learn what makes someone likely to default.

--

## What if the number is higher or lower?
* Higher (ex: 20): you are telling the model **defaulters matter even more**. It will try very hard to catch every defaulter but will also flag many non-defaulters incorrectly. More false alarms.
* Lower (ex: 2): you are barely correcting the imbalance. The model still leans toward predicting "will repay" for most people. More missed defaulters.

11.39 is the exact ratio from your data, not a judgment call. It is the mathematically neutral correction that says: treat both classes as if they appeared equally.

# 5: Model training

In [ ]:
model.fit(X_train, y_train)
print("Training complete.")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:09:12] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Training complete.


# 6: Model evaluation

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report

y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print("ROC-AUC:", round(roc_auc_score(y_test, y_pred_proba), 4))
print()
print(classification_report(y_test, y_pred))

ROC-AUC: 0.7515

              precision    recall  f1-score   support

           0       0.96      0.69      0.81     56538
           1       0.16      0.68      0.26      4965

    accuracy                           0.69     61503
   macro avg       0.56      0.69      0.53     61503
weighted avg       0.90      0.69      0.76     61503



### Line-by-line exp

* `model.predict_proba(X_test)[:, 1]` -- asks the model: for each applicant in the test set, what is the probability they will default? The `[:, 1]` extracts the probability for class 1 (default) specifically.
* `model.predict(X_test)` -- converts those probabilities into hard predictions (0 or 1) using a 0.5 threshold.
* `roc_auc_score` -- calculates the ROC-AUC from the probabilities.
* `classification_report` -- calculates precision, recall, F1 from the hard predictions.